# ASAP7y event-based SNR across VIP dendritic ROIs

This notebook replaces the earlier full-session dynamic-range and local-MAD metrics with a **simpler event-based SNR** that is closer to how SNR is reported in the supplied imaging-methods documentation.

The main idea is:

\[
\mathrm{SNR}_{event} = \frac{\mathrm{event\ amplitude}}{\sigma_{background}}
\]

where:

- **event amplitude** is the peak dF/F of detected fluorescence/voltage events after local drift removal;
- **background noise** is the standard deviation of the trace **outside detected events**, after removing slow bleaching/drift with a linear fit;
- ROI-level SNR is summarized as the median, mean, and high-percentile event SNR across detected events.

This follows the spirit of Neugornet et al. 2021, who define SNR by dividing event amplitude by the background noise standard deviation of the event-excluded trace. It also follows Gharia et al. 2020 in treating SNR as a background-subtracted signal relative to measured background variation, rather than as a sample-to-sample fast-noise estimate.

For ASAP7y voltage imaging, this is not a single biophysical ground-truth SNR. It is a practical **detected-event SNR**: "how large are prominent voltage-associated dF/F excursions relative to the local non-event background variability?"

In [1]:
import os
import sys
import glob
import json
import h5py
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks, peak_widths, find_peaks_cwt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.extraction import load_voltage_roi_transform_h5
from vip_slap2_analysis.voltage import analysis
from vip_slap2_analysis.common.qc import robust_sigma
from vip_slap2_analysis.plotting.plot_psth import plot_voltage_mean_image_response_heatmap
from vip_slap2_analysis.utils.utils import save_figure

import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    import seaborn as sns
    sns.set_style("white")
except Exception:
    sns = None

params = {
    "legend.fontsize": "large",
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "large",
    "ytick.labelsize": "large",
}
plt.rcParams.update(params)

display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

## Build session registry

In [3]:
today_str = datetime.today().strftime("%Y-%m-%d")

BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(
    r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots"
)

TARGET_MICE = [
    826031,
    826032,
]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
# This variant controls the filename suffix. The plotted dataset is SIGNAL below.
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"      # one of: "raw_f", "f0", "dff"

# Optional direct override. Leave as None to resolve from asset.derived_dir / "voltage".
SESSION_TRACE_H5 = None

SAVE_PATH.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {SAVE_PATH}")

Output directory: C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots


In [4]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions.")
display(process_df)

Found 19 candidate sessions.


,session_id,subject_id,session_#,session_date,indicator1,indicator2,dmd1_depth,dmd2_depth,paradigm,session_type,...,instrument_id,camera_type,has raster ROI?,has integration roi?,behavior_rig,quality,flags,session_dir,purpose,notes
0,826031_2026-01-30_15-04-02,826031,2,2026-01-30,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
1,826031_2026-02-01_11-01-50,826031,3,2026-02-01,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
2,826031_2026-02-02_10-23-53,826031,4,2026-02-02,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
3,826031_2026-02-03_14-21-45,826031,5,2026-02-03,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
4,826031_2026-02-04_12-15-34,826031,6,2026-02-04,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
5,826031_2026-02-05_09-28-56,826031,7,2026-02-05,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
6,826031_2026-02-06_10-21-20,826031,8,2026-02-06,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
7,826031_2026-02-10_10-48-51,826031,9,2026-02-10,ASAP7y,NaN,25,250,change_detection_passive,novel,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
8,826031_2026-02-11_12-42-21,826031,10,2026-02-11,ASAP7y,NaN,25,250,change_detection_passive,novel+,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
9,826031_2026-02-12_07-43-37,826031,11,2026-02-12,ASAP7y,NaN,25,250,change_detection_passive,novel+,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN


## Method implemented here

### Primary metric: detected-event SNR

For each ROI and each sampled trace window:

1. Read dF/F and optionally average/downsample it to a biological analysis rate.
2. Lightly smooth the trace on a short voltage/optical timescale.
3. Find candidate positive events. Because the processed ASAP dF/F is expected to be inverted upstream, positive-going dF/F excursions are treated as voltage-associated events. Set `EVENT_POLARITY = "negative"` or `"both"` if needed.
4. Exclude event windows.
5. Fit a linear drift/bleaching baseline to non-event samples only.
6. Recompute event amplitudes after subtracting that baseline.
7. Estimate background noise as the standard deviation of non-event residuals.
8. Compute event SNR:

\[
\mathrm{SNR}_{event} = \frac{A_{event}}{\sigma_{background}}
\]

The notebook also reports a squared-power version:

\[
\mathrm{SNR}_{power} = \frac{A_{event}^2}{\sigma_{background}^2}
\]

but the slide-friendly quantity is the linear event SNR.

### Why this is preferable here

This avoids comparing whole-session drift to sample-to-sample noise. The denominator is not a sub-millisecond electronic noise estimate; it is the variability of the non-event trace background on the same preprocessed trace used to detect events.

In [5]:
# Example-session display
EXAMPLE_ASSET_INDEX = 0
TRACE_START_SEC = 30       # None -> begin 30 s after trace start
TRACE_DURATION_SEC = 30

# Event-SNR estimation
ANALYSIS_RATE_HZ = 1000.0    # block-average from ~10.8 kHz to this rate before event detection; set None to keep native
EVENT_SMOOTH_MS = 5.0        # light smoothing before detection; 3-10 ms is useful for voltage dF/F
METRIC_WINDOW_SEC = 20.0     # local windows used for event/noise estimation
N_METRIC_WINDOWS = 40        # distributed across each recording
MIN_VALID_SAMPLES = 500
MIN_EVENTS_FOR_SNR = 2

# Event detector parameters
EVENT_POLARITY = "positive"  # "positive", "negative", or "both"; canonical inverted ASAP dF/F should be positive
DETECTOR = "peaks"           # "peaks" or "wavelet_cwt"; peaks is faster and more transparent
EVENT_THRESHOLD_SD = 3.0     # minimum peak height in background SD units
EVENT_PROMINENCE_SD = 2.0    # minimum peak prominence in background SD units
MIN_PEAK_DISTANCE_MS = 20.0
MIN_EVENT_WIDTH_MS = 2.0
MAX_EVENT_WIDTH_MS = 500.0
EVENT_EXCLUDE_PAD_MS = 25.0  # extra mask around each event when estimating background

# CWT detector settings, used only if DETECTOR = "wavelet_cwt"
CWT_N_WIDTHS = 24
CWT_MIN_SNR = 1.5

# Plotting / output
SNR_PLOT_METRIC = "event_snr_median"  # one of event_snr_median, event_snr_mean, event_snr_p90
HIST_BINS = 30
SCATTER_ALPHA = 0.65
SAVE_FIGURES = True
SAVE_TABLE = True

print("Primary metric:", SNR_PLOT_METRIC)

Primary metric: event_snr_median


## HDF5 and session helpers

In [6]:
def resolve_trace_h5(asset, trace_variant=TRACE_VARIANT, override=SESSION_TRACE_H5):
    """Resolve the session-long voltage trace file while retaining the asset scheme."""
    if override is not None:
        path = Path(override)
    else:
        path = Path(asset.derived_dir) / "voltage" / f"voltage_session_traces_{trace_variant}.h5"

    if path.exists():
        return path

    # Helpful fallbacks in case naming changed.
    candidates = sorted((Path(asset.derived_dir) / "voltage").glob("voltage_session_traces*.h5"))
    if candidates:
        warnings.warn(f"Could not find expected file {path.name}; using {candidates[0].name}")
        return candidates[0]

    raise FileNotFoundError(f"No session trace H5 found for asset {asset} at {path}")


def _decode_array(values):
    out = []
    for v in values:
        if isinstance(v, bytes):
            out.append(v.decode())
        else:
            out.append(str(v))
    return out


def get_roi_ids(group, n_roi):
    if "roi_ids" in group:
        return _decode_array(group["roi_ids"][:])
    return [f"roi{i:04d}" for i in range(n_roi)]


def get_valid_roi_mask(group, n_roi):
    for key in ["valid_roi", "valid_rois", "roi_valid", "is_valid"]:
        if key in group:
            v = np.asarray(group[key][:]).astype(bool)
            if v.size == n_roi:
                return v
    return np.ones(n_roi, dtype=bool)


def get_timebase(group):
    for key in ["timebase_sec", "time_sec", "timestamps", "time"]:
        if key in group:
            return np.asarray(group[key][:], dtype=float)
    return None


def estimate_sampling_rate(group, fallback=10800.0):
    t = get_timebase(group)
    if t is None or len(t) < 10:
        return float(fallback)
    # Sample the timebase in case it is huge.
    idx = np.linspace(0, len(t) - 1, min(len(t), 5000)).astype(int)
    dt = np.diff(t[idx])
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return float(fallback)
    # Because idx may skip samples, divide by the index step.
    didx = np.diff(idx)
    didx = didx[:len(dt)]
    sample_dt = dt / didx
    return float(1.0 / np.nanmedian(sample_dt))


def infer_trace_orientation(ds, group):
    """Return 'roi_time' or 'time_roi'."""
    t = get_timebase(group)
    shape = ds.shape
    if len(shape) != 2:
        raise ValueError(f"Expected 2-D trace dataset, found shape {shape}")

    if t is not None:
        nt = len(t)
        if shape[1] == nt:
            return "roi_time"
        if shape[0] == nt:
            return "time_roi"

    # Most voltage files are ROI x time; fall back to smaller dimension as ROI.
    return "roi_time" if shape[0] < shape[1] else "time_roi"


def get_trace_shape(group, signal=SIGNAL):
    ds = group[signal]
    orient = infer_trace_orientation(ds, group)
    if orient == "roi_time":
        n_roi, n_time = ds.shape
    else:
        n_time, n_roi = ds.shape
    return n_roi, n_time, orient


def read_trace_window(group, signal, start_idx, stop_idx, roi_indices=None):
    """Read a trace window as ROI x time."""
    ds = group[signal]
    n_roi, n_time, orient = get_trace_shape(group, signal)

    start_idx = int(max(0, start_idx))
    stop_idx = int(min(n_time, stop_idx))
    if roi_indices is None:
        roi_indices = np.arange(n_roi)
    roi_indices = np.asarray(roi_indices, dtype=int)

    if orient == "roi_time":
        X = ds[roi_indices, start_idx:stop_idx]
    else:
        X = ds[start_idx:stop_idx, roi_indices].T

    return np.asarray(X, dtype=np.float32)


def get_dmd_keys(h5):
    return sorted(k for k in h5.keys() if k.upper().startswith("DMD"))


def asset_metadata_value(asset, key, default=np.nan):
    try:
        return asset.metadata.get(key, default)
    except Exception:
        return default


def asset_session_id(asset):
    for attr in ["session_id", "session_name", "session"]:
        if hasattr(asset, attr):
            return getattr(asset, attr)
    return asset_metadata_value(asset, "session_id", "unknown_session")


def asset_subject_id(asset):
    for attr in ["subject_id", "mouse_id"]:
        if hasattr(asset, attr):
            return getattr(asset, attr)
    return asset_metadata_value(asset, "subject_id", asset_metadata_value(asset, "mouse_id", np.nan))

## Event-based SNR helpers

In [7]:
def _nan_interpolate_1d(x):
    """Fill NaNs by linear interpolation for filtering/detection."""
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if finite.all():
        return x
    if finite.sum() < 2:
        return np.full_like(x, np.nan)
    idx = np.arange(x.size)
    y = x.copy()
    y[~finite] = np.interp(idx[~finite], idx[finite], x[finite])
    return y


def _mad_sigma(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    med = np.nanmedian(x)
    return 1.4826 * np.nanmedian(np.abs(x - med))


def _smooth_1d(x, fs, smooth_ms):
    x = _nan_interpolate_1d(x)
    if not np.isfinite(x).any():
        return x
    n = int(round(float(smooth_ms) * 1e-3 * fs))
    n = max(1, n)
    if n <= 1:
        return x
    return uniform_filter1d(x, size=n, mode="nearest")


def _block_average_2d(X, factor):
    """Block-average ROI x time data along time."""
    factor = int(max(1, factor))
    if factor <= 1:
        return X
    n = X.shape[1]
    n_trim = (n // factor) * factor
    if n_trim < factor:
        return X
    X = X[:, :n_trim]
    return np.nanmean(X.reshape(X.shape[0], n_trim // factor, factor), axis=2)


def _linear_fit_baseline(y, mask):
    """Fit y ~ t using mask; return fitted baseline across all samples."""
    y = np.asarray(y, dtype=float)
    t = np.arange(y.size, dtype=float)
    fit_mask = np.asarray(mask, dtype=bool) & np.isfinite(y)
    if fit_mask.sum() < 5:
        # Fall back to median baseline when there are too few background samples.
        med = np.nanmedian(y)
        return np.full_like(y, med, dtype=float)

    # Center/scale t for numerical stability.
    tc = (t - np.nanmean(t[fit_mask])) / max(1.0, np.nanstd(t[fit_mask]))
    coef = np.polyfit(tc[fit_mask], y[fit_mask], deg=1)
    return np.polyval(coef, tc)


def _event_signal_for_polarity(y, polarity):
    if polarity == "positive":
        return y
    if polarity == "negative":
        return -y
    if polarity == "both":
        return np.abs(y)
    raise ValueError("EVENT_POLARITY must be 'positive', 'negative', or 'both'.")


def _find_candidate_peaks(y_signal, fs, sigma, detector=DETECTOR):
    """Find event candidate peaks on a positive signal."""
    if not np.isfinite(sigma) or sigma <= 0:
        return np.asarray([], dtype=int), {}

    min_distance = max(1, int(round(MIN_PEAK_DISTANCE_MS * 1e-3 * fs)))
    height = EVENT_THRESHOLD_SD * sigma
    prominence = EVENT_PROMINENCE_SD * sigma

    if detector == "peaks":
        peaks, props = find_peaks(
            y_signal,
            height=height,
            prominence=prominence,
            distance=min_distance,
        )
        return peaks.astype(int), props

    if detector == "wavelet_cwt":
        min_w = max(1, MIN_EVENT_WIDTH_MS * 1e-3 * fs)
        max_w = max(min_w + 1, MAX_EVENT_WIDTH_MS * 1e-3 * fs)
        widths = np.unique(np.round(np.geomspace(min_w, max_w, CWT_N_WIDTHS)).astype(int))
        peaks = find_peaks_cwt(
            y_signal,
            widths,
            min_snr=CWT_MIN_SNR,
        )
        peaks = np.asarray(peaks, dtype=int)

        # Apply the same transparent amplitude/prominence filters after CWT candidate generation.
        if peaks.size:
            keep = np.isfinite(y_signal[peaks]) & (y_signal[peaks] >= height)
            peaks = peaks[keep]
            if peaks.size:
                p2, props = find_peaks(y_signal, height=height, prominence=prominence, distance=min_distance)
                # Keep CWT peaks that are close to an amplitude/prominence peak.
                tol = max(1, int(round(0.5 * min_distance)))
                matched = []
                for p in peaks:
                    if np.any(np.abs(p2 - p) <= tol):
                        matched.append(p2[np.argmin(np.abs(p2 - p))])
                peaks = np.unique(np.asarray(matched, dtype=int))
                return peaks, props
        return np.asarray([], dtype=int), {}

    raise ValueError("DETECTOR must be 'peaks' or 'wavelet_cwt'.")


def _make_event_mask(y_signal, peaks, fs):
    """Mask samples around events for background/noise estimation."""
    n = len(y_signal)
    mask = np.zeros(n, dtype=bool)
    if len(peaks) == 0:
        return mask

    try:
        widths, _, left_ips, right_ips = peak_widths(y_signal, peaks, rel_height=0.5)
    except Exception:
        widths = np.full(len(peaks), max(1, int(round(MIN_EVENT_WIDTH_MS * 1e-3 * fs))))
        left_ips = peaks - widths / 2
        right_ips = peaks + widths / 2

    min_w = int(round(MIN_EVENT_WIDTH_MS * 1e-3 * fs))
    max_w = int(round(MAX_EVENT_WIDTH_MS * 1e-3 * fs))
    pad = int(round(EVENT_EXCLUDE_PAD_MS * 1e-3 * fs))

    for p, w, left, right in zip(peaks, widths, left_ips, right_ips):
        w = int(np.clip(round(w), min_w, max_w))
        i0 = int(max(0, min(left, p - w // 2) - pad))
        i1 = int(min(n, max(right, p + w // 2) + pad))
        if i1 > i0:
            mask[i0:i1] = True
    return mask


def event_snr_for_trace(
    x,
    fs,
    *,
    smooth_ms=EVENT_SMOOTH_MS,
    polarity=EVENT_POLARITY,
    detector=DETECTOR,
):
    """
    Estimate event SNR for one trace window.

    Returns a dict plus arrays of event amplitudes/SNRs. The implementation
    matches the documentation conceptually: detect events, define background as
    non-event trace portions, remove slow drift/bleaching by regression, then
    divide event amplitudes by background-noise SD.
    """
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if finite.sum() < MIN_VALID_SAMPLES:
        return {
            "n_events": 0,
            "duration_sec": finite.sum() / fs,
            "background_noise_sd": np.nan,
            "background_noise_robust": np.nan,
            "event_amplitudes": np.asarray([]),
            "event_snrs": np.asarray([]),
        }

    xs = _smooth_1d(x, fs, smooth_ms=smooth_ms)

    # Initial drift removal using all valid points.
    baseline0 = _linear_fit_baseline(xs, np.isfinite(xs))
    y0 = xs - baseline0
    sigma0 = _mad_sigma(y0)
    if not np.isfinite(sigma0) or sigma0 <= 0:
        return {
            "n_events": 0,
            "duration_sec": finite.sum() / fs,
            "background_noise_sd": np.nan,
            "background_noise_robust": np.nan,
            "event_amplitudes": np.asarray([]),
            "event_snrs": np.asarray([]),
        }

    y0_signal = _event_signal_for_polarity(y0, polarity)
    peaks0, _ = _find_candidate_peaks(y0_signal, fs, sigma0, detector=detector)
    event_mask0 = _make_event_mask(y0_signal, peaks0, fs)
    bg_mask0 = np.isfinite(xs) & ~event_mask0

    # Final drift/bleaching baseline fit only to non-event background, as in the documentation.
    baseline = _linear_fit_baseline(xs, bg_mask0)
    resid = xs - baseline

    bg_values = resid[bg_mask0 & np.isfinite(resid)]
    if bg_values.size < MIN_VALID_SAMPLES:
        bg_values = resid[np.isfinite(resid)]

    bg_noise_sd = float(np.nanstd(bg_values, ddof=1)) if bg_values.size > 1 else np.nan
    bg_noise_robust = float(_mad_sigma(bg_values))

    if not np.isfinite(bg_noise_sd) or bg_noise_sd <= 0:
        return {
            "n_events": 0,
            "duration_sec": finite.sum() / fs,
            "background_noise_sd": bg_noise_sd,
            "background_noise_robust": bg_noise_robust,
            "event_amplitudes": np.asarray([]),
            "event_snrs": np.asarray([]),
        }

    y_signal = _event_signal_for_polarity(resid, polarity)
    peaks, props = _find_candidate_peaks(y_signal, fs, bg_noise_sd, detector=detector)
    event_mask = _make_event_mask(y_signal, peaks, fs)

    # Refit once more after final event mask.
    bg_mask = np.isfinite(xs) & ~event_mask
    baseline = _linear_fit_baseline(xs, bg_mask)
    resid = xs - baseline
    y_signal = _event_signal_for_polarity(resid, polarity)

    bg_values = resid[bg_mask & np.isfinite(resid)]
    bg_noise_sd = float(np.nanstd(bg_values, ddof=1)) if bg_values.size > 1 else np.nan
    bg_noise_robust = float(_mad_sigma(bg_values))
    if not np.isfinite(bg_noise_sd) or bg_noise_sd <= 0:
        event_amplitudes = np.asarray([])
        event_snrs = np.asarray([])
    else:
        event_amplitudes = y_signal[peaks]
        event_amplitudes = event_amplitudes[np.isfinite(event_amplitudes) & (event_amplitudes > 0)]
        event_snrs = event_amplitudes / bg_noise_sd

    return {
        "n_events": int(event_snrs.size),
        "duration_sec": finite.sum() / fs,
        "background_noise_sd": bg_noise_sd,
        "background_noise_robust": bg_noise_robust,
        "event_amplitudes": event_amplitudes,
        "event_snrs": event_snrs,
    }


def summarize_event_snr(event_amplitudes, event_snrs, bg_noise_values, n_events_total, duration_total):
    """Summarize pooled event/background measurements for one ROI."""
    event_amplitudes = np.asarray(event_amplitudes, dtype=float)
    event_snrs = np.asarray(event_snrs, dtype=float)
    bg_noise_values = np.asarray(bg_noise_values, dtype=float)
    event_amplitudes = event_amplitudes[np.isfinite(event_amplitudes)]
    event_snrs = event_snrs[np.isfinite(event_snrs)]
    bg_noise_values = bg_noise_values[np.isfinite(bg_noise_values) & (bg_noise_values > 0)]

    row = {
        "n_events": int(n_events_total),
        "event_rate_per_min": 60.0 * n_events_total / duration_total if duration_total > 0 else np.nan,
        "background_noise_sd_dff": np.nanmedian(bg_noise_values) if bg_noise_values.size else np.nan,
        "event_amplitude_median_dff": np.nanmedian(event_amplitudes) if event_amplitudes.size else np.nan,
        "event_amplitude_mean_dff": np.nanmean(event_amplitudes) if event_amplitudes.size else np.nan,
        "event_amplitude_p90_dff": np.nanpercentile(event_amplitudes, 90) if event_amplitudes.size else np.nan,
        "event_snr_median": np.nanmedian(event_snrs) if event_snrs.size else np.nan,
        "event_snr_mean": np.nanmean(event_snrs) if event_snrs.size else np.nan,
        "event_snr_p90": np.nanpercentile(event_snrs, 90) if event_snrs.size else np.nan,
        "event_snr_power_median": np.nanmedian(event_snrs ** 2) if event_snrs.size else np.nan,
        "event_snr_db_median": 20.0 * np.log10(np.nanmedian(event_snrs)) if event_snrs.size and np.nanmedian(event_snrs) > 0 else np.nan,
    }
    return row

## Estimate ROI event SNR for a session

In [8]:
def distributed_window_starts(n_time, win_n, n_windows):
    """Evenly distribute window starts across a recording."""
    win_n = int(win_n)
    if n_time <= win_n:
        return np.asarray([0], dtype=int)

    max_start = n_time - win_n
    n_windows = int(max(1, min(n_windows, max_start + 1)))
    starts = np.linspace(0, max_start, n_windows)
    starts = np.unique(np.round(starts).astype(int))
    return starts


def estimate_dmd_event_snr(
    group,
    *,
    signal=SIGNAL,
    analysis_rate_hz=ANALYSIS_RATE_HZ,
    metric_window_sec=METRIC_WINDOW_SEC,
    n_metric_windows=N_METRIC_WINDOWS,
):
    n_roi, n_time, orient = get_trace_shape(group, signal)
    fs_native = estimate_sampling_rate(group)
    down_factor = 1
    fs = fs_native
    if analysis_rate_hz is not None and analysis_rate_hz > 0:
        down_factor = max(1, int(round(fs_native / analysis_rate_hz)))
        fs = fs_native / down_factor

    win_n_native = int(round(metric_window_sec * fs_native))
    starts = distributed_window_starts(n_time, win_n_native, n_metric_windows)

    roi_ids = get_roi_ids(group, n_roi)
    valid_rois = get_valid_roi_mask(group, n_roi)

    events_by_roi = [[] for _ in range(n_roi)]
    snrs_by_roi = [[] for _ in range(n_roi)]
    bg_noise_by_roi = [[] for _ in range(n_roi)]
    duration_by_roi = np.zeros(n_roi, dtype=float)

    for widx, start in enumerate(starts):
        stop = min(n_time, int(start + win_n_native))
        X = read_trace_window(group, signal, start, stop)
        if down_factor > 1:
            X = _block_average_2d(X, down_factor)

        for r in range(n_roi):
            if not valid_rois[r]:
                continue
            result = event_snr_for_trace(X[r], fs)
            duration_by_roi[r] += result["duration_sec"]
            if np.isfinite(result["background_noise_sd"]) and result["background_noise_sd"] > 0:
                bg_noise_by_roi[r].append(result["background_noise_sd"])
            if result["n_events"] > 0:
                events_by_roi[r].extend(result["event_amplitudes"])
                snrs_by_roi[r].extend(result["event_snrs"])

    rows = []
    for r in range(n_roi):
        summary = summarize_event_snr(
            np.asarray(events_by_roi[r], dtype=float),
            np.asarray(snrs_by_roi[r], dtype=float),
            np.asarray(bg_noise_by_roi[r], dtype=float),
            n_events_total=len(snrs_by_roi[r]),
            duration_total=duration_by_roi[r],
        )
        is_valid = bool(
            valid_rois[r]
            and summary["n_events"] >= MIN_EVENTS_FOR_SNR
            and np.isfinite(summary[SNR_PLOT_METRIC])
        )
        rows.append({
            "roi_index": r,
            "roi_id": roi_ids[r],
            "valid_roi": is_valid,
            "raw_valid_roi": bool(valid_rois[r]),
            "sampling_rate_hz_native": fs_native,
            "analysis_rate_hz": fs,
            "downsample_factor": down_factor,
            "metric_window_sec": metric_window_sec,
            "n_metric_windows": len(starts),
            "event_smooth_ms": EVENT_SMOOTH_MS,
            "event_polarity": EVENT_POLARITY,
            "detector": DETECTOR,
            "event_threshold_sd": EVENT_THRESHOLD_SD,
            "event_prominence_sd": EVENT_PROMINENCE_SD,
            **summary,
        })

    return pd.DataFrame(rows)


def session_event_snr_table(asset):
    """Calculate event-SNR metrics for every DMD in one session."""
    h5_path = resolve_trace_h5(asset)
    tables = []
    with h5py.File(h5_path, "r") as h5:
        for dmd in get_dmd_keys(h5):
            if SIGNAL not in h5[dmd]:
                warnings.warn(f"{h5_path.name}: {dmd} has no dataset '{SIGNAL}'. Skipping.")
                continue
            tab = estimate_dmd_event_snr(h5[dmd])
            tab.insert(0, "dmd", dmd)
            tables.append(tab)

    if not tables:
        return pd.DataFrame()

    out = pd.concat(tables, ignore_index=True)
    out.insert(0, "session_id", asset_session_id(asset))
    out.insert(1, "subject_id", asset_subject_id(asset))
    out.insert(2, "h5_path", str(h5_path))

    # Add depth metadata if present.
    depth_map = {
        "DMD1": asset_metadata_value(asset, "dmd1_depth", np.nan),
        "DMD2": asset_metadata_value(asset, "dmd2_depth", np.nan),
    }
    out["depth_um_below_pia"] = out["dmd"].map(depth_map)

    return out


example_asset = assets[EXAMPLE_ASSET_INDEX]
example_snr_df = session_event_snr_table(example_asset)

print(f"Example session: {asset_session_id(example_asset)}")
display(example_snr_df.sort_values(SNR_PLOT_METRIC, ascending=False))

Example session: 826031_2026-01-30_15-04-02


,session_id,subject_id,h5_path,dmd,roi_index,roi_id,valid_roi,raw_valid_roi,sampling_rate_hz_native,analysis_rate_hz,...,background_noise_sd_dff,event_amplitude_median_dff,event_amplitude_mean_dff,event_amplitude_p90_dff,event_snr_median,event_snr_mean,event_snr_p90,event_snr_power_median,event_snr_db_median,depth_um_below_pia
30,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD2,15,DMD2_roi0015,True,True,10724.166028,974.924184,...,0.091045,0.469788,0.467412,0.537930,5.266189,5.322237,6.690815,27.732756,14.429930,250
16,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD2,1,DMD2_roi0001,True,True,10724.166028,974.924184,...,0.105260,0.359156,0.367461,0.441585,3.750579,3.827010,4.696253,14.066845,11.481967,250
4,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD1,4,DMD1_roi0004,True,True,10724.166028,974.924184,...,0.150287,0.516248,0.494806,0.547105,3.725985,3.615504,3.967407,13.882961,11.424821,25
3,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD1,3,DMD1_roi0003,True,True,10724.166028,974.924184,...,0.104559,0.348965,0.350849,0.422235,3.638435,3.654954,4.107810,13.238242,11.218292,25
11,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD1,11,DMD1_roi0011,True,True,10724.166028,974.924184,...,0.175124,0.570156,0.570704,0.646967,3.604314,3.582827,3.894720,12.991106,11.136453,25
17,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD2,2,DMD2_roi0002,True,True,10724.166028,974.924184,...,0.162505,0.610649,0.582230,0.618722,3.603579,3.509648,3.677182,12.985780,11.134680,250
7,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD1,7,DMD1_roi0007,True,True,10724.166028,974.924184,...,0.082324,0.261874,0.261778,0.297289,3.572554,3.623675,4.101263,12.763153,11.059575,25
1,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD1,1,DMD1_roi0001,True,True,10724.166028,974.924184,...,0.111790,0.352089,0.350431,0.393004,3.552513,3.548472,3.901320,12.620349,11.010714,25
25,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD2,10,DMD2_roi0010,True,True,10724.166028,974.924184,...,0.118654,0.359680,0.378601,0.451420,3.545126,3.554258,3.996420,12.567918,10.992633,250
29,826031_2026-01-30_15-04-02,826031,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,DMD2,14,DMD2_roi0014,True,True,10724.166028,974.924184,...,0.145778,0.447453,0.449452,0.562065,3.490826,3.564195,4.055295,12.185864,10.858563,250


## 1. Example session: plot all ROI snippets sorted by event SNR

In [9]:
def _segment_start_stop_from_seconds(group, start_sec, duration_sec):
    fs = estimate_sampling_rate(group)
    n_roi, n_time, orient = get_trace_shape(group, SIGNAL)
    t = get_timebase(group)
    if t is None:
        if start_sec is None:
            start_idx = int(round(30 * fs))
        else:
            start_idx = int(round(start_sec * fs))
    else:
        if start_sec is None:
            start_time = float(t[0]) + 30.0
        else:
            start_time = float(start_sec)
        start_idx = int(np.searchsorted(t, start_time))
    stop_idx = int(min(n_time, start_idx + round(duration_sec * fs)))
    start_idx = int(max(0, min(start_idx, n_time - 1)))
    return start_idx, stop_idx


def plot_snr_ranked_trace_segment(
    asset,
    snr_df,
    *,
    start_sec=TRACE_START_SEC,
    duration_sec=TRACE_DURATION_SEC,
    metric=SNR_PLOT_METRIC,
):
    h5_path = resolve_trace_h5(asset)

    with h5py.File(h5_path, "r") as h5:
        dmd_keys = get_dmd_keys(h5)
        fig, axes = plt.subplots(
            len(dmd_keys),
            1,
            figsize=(14, max(4, 0.32 * len(snr_df) + 2)),
            sharex=False,
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)

        for ax, dmd in zip(axes, dmd_keys):
            group = h5[dmd]
            fs = estimate_sampling_rate(group)
            i0, i1 = _segment_start_stop_from_seconds(group, start_sec, duration_sec)
            X = read_trace_window(group, SIGNAL, i0, i1)

            tbase = get_timebase(group)
            if tbase is None:
                t = np.arange(X.shape[1]) / fs
            else:
                t = tbase[i0:i1]
                t = t - t[0]

            sub = snr_df.query("dmd == @dmd").copy()
            sub = sub.sort_values(metric, ascending=False, na_position="last")
            order = sub["roi_index"].to_numpy(dtype=int)
            X = X[order]

            # Same amplitude scale for all traces within a DMD panel.
            X0 = X - np.nanmedian(X, axis=1, keepdims=True)
            scale = np.nanpercentile(np.abs(X0), 99)
            if not np.isfinite(scale) or scale <= 0:
                scale = np.nanstd(X0)
            if not np.isfinite(scale) or scale <= 0:
                scale = 1.0

            offset = 3.0 * scale
            for j, (_, row) in enumerate(sub.iterrows()):
                y = X0[j] + (len(sub) - 1 - j) * offset
                lw = 1.0 if bool(row["valid_roi"]) else 0.7
                alpha = 0.95 if bool(row["valid_roi"]) else 0.35
                ax.plot(t, y, lw=lw, alpha=alpha)
                label = f"{row['roi_id']} | SNR={row[metric]:.1f} | n={int(row['n_events'])}" if np.isfinite(row[metric]) else f"{row['roi_id']} | no events"
                ax.text(
                    t[-1] + 0.01 * (t[-1] - t[0]),
                    (len(sub) - 1 - j) * offset,
                    label,
                    va="center",
                    fontsize=8,
                )

            ax.set_title(f"{asset_session_id(asset)} {dmd}: all ROIs sorted by {metric}")
            ax.set_ylabel(f"{SIGNAL} + offset")
            ax.spines[["right", "top"]].set_visible(False)
            ax.set_yticks([])
            ax.set_xlim(t[0], t[-1] + 0.25 * (t[-1] - t[0]))
            ax.axhline(-offset, lw=0.8, color="0.6", alpha=0.4)

        axes[-1].set_xlabel("Time in plotted segment (s)")

    return fig


fig = plot_snr_ranked_trace_segment(example_asset, example_snr_df)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_event_snr_ranked_traces_example.png", dpi=300, bbox_inches="tight")
plt.show()

<IPython.core.display.Javascript object>

## Optional sanity check: show detected events on selected ROIs

In [10]:
def plot_event_detection_sanity(
    asset,
    snr_df,
    *,
    dmd=None,
    n_examples=6,
    start_sec=TRACE_START_SEC,
    duration_sec=TRACE_DURATION_SEC,
    metric=SNR_PLOT_METRIC,
):
    h5_path = resolve_trace_h5(asset)
    if dmd is None:
        dmd = sorted(snr_df["dmd"].unique())[0]

    sub = snr_df.query("dmd == @dmd and valid_roi").sort_values(metric, ascending=False)
    if len(sub) == 0:
        print("No valid ROIs to plot.")
        return None

    # High, middle, low examples.
    idx = np.linspace(0, len(sub) - 1, min(n_examples, len(sub))).round().astype(int)
    sub = sub.iloc[idx]

    with h5py.File(h5_path, "r") as h5:
        group = h5[dmd]
        fs_native = estimate_sampling_rate(group)
        down_factor = max(1, int(round(fs_native / ANALYSIS_RATE_HZ))) if ANALYSIS_RATE_HZ else 1
        fs = fs_native / down_factor
        i0, i1 = _segment_start_stop_from_seconds(group, start_sec, duration_sec)

        fig, axes = plt.subplots(len(sub), 1, figsize=(14, 2.2 * len(sub)), sharex=True, constrained_layout=True)
        axes = np.atleast_1d(axes)

        for ax, (_, row) in zip(axes, sub.iterrows()):
            X = read_trace_window(group, SIGNAL, i0, i1, roi_indices=[int(row["roi_index"])])
            if down_factor > 1:
                X = _block_average_2d(X, down_factor)
            x = X[0]
            xs = _smooth_1d(x, fs, EVENT_SMOOTH_MS)

            # Re-run detection for this plotted segment.
            baseline0 = _linear_fit_baseline(xs, np.isfinite(xs))
            y0 = xs - baseline0
            sigma0 = _mad_sigma(y0)
            peaks0, _ = _find_candidate_peaks(_event_signal_for_polarity(y0, EVENT_POLARITY), fs, sigma0)
            mask0 = _make_event_mask(_event_signal_for_polarity(y0, EVENT_POLARITY), peaks0, fs)
            baseline = _linear_fit_baseline(xs, np.isfinite(xs) & ~mask0)
            resid = xs - baseline
            bg = resid[np.isfinite(resid) & ~mask0]
            sigma = np.nanstd(bg, ddof=1) if bg.size > 1 else np.nan
            signal = _event_signal_for_polarity(resid, EVENT_POLARITY)
            peaks, _ = _find_candidate_peaks(signal, fs, sigma)

            t = np.arange(len(resid)) / fs
            ax.plot(t, resid, lw=1.0, color="0.15")
            if len(peaks):
                ax.scatter(t[peaks], resid[peaks], s=25, zorder=3, label="detected events")
            if np.isfinite(sigma):
                ax.axhline(EVENT_THRESHOLD_SD * sigma, ls="--", lw=0.9, color="0.35", label=f"{EVENT_THRESHOLD_SD:g}σ")
                if EVENT_POLARITY in ["negative", "both"]:
                    ax.axhline(-EVENT_THRESHOLD_SD * sigma, ls="--", lw=0.9, color="0.35")
            ax.set_ylabel(row["roi_id"])
            ax.set_title(f"{dmd} {row['roi_id']} | session median SNR={row[metric]:.1f} | session n={int(row['n_events'])}")
            ax.spines[["right", "top"]].set_visible(False)
            ax.legend(frameon=False, loc="upper right")

        axes[-1].set_xlabel("Time in plotted segment (s)")

    return fig


fig = plot_event_detection_sanity(example_asset, example_snr_df, dmd="DMD1")
if SAVE_FIGURES and fig is not None:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_event_detection_sanity_example.png", dpi=300, bbox_inches="tight")
plt.show()

<IPython.core.display.Javascript object>

## 2. Estimate event SNR across all sessions / mice

In [ ]:
all_tables = []
errors = []

for i, asset in enumerate(assets):
    sid = asset_session_id(asset)
    print(f"[{i + 1}/{len(assets)}] {sid}")
    try:
        tab = session_event_snr_table(asset)
        if len(tab):
            all_tables.append(tab)
            print(f"  {len(tab)} ROIs; {tab['valid_roi'].sum()} valid with >= {MIN_EVENTS_FOR_SNR} events")
        else:
            print("  No rows returned.")
    except Exception as exc:
        warnings.warn(f"Failed {sid}: {exc}")
        errors.append({"session_id": sid, "error": repr(exc)})

snr_df = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
valid_snr_df = snr_df.query("valid_roi").copy() if len(snr_df) else pd.DataFrame()
error_df = pd.DataFrame(errors)

print(f"\nTotal ROIs: {len(snr_df)}")
print(f"Valid ROIs with event SNR: {len(valid_snr_df)}")
if len(error_df):
    display(error_df)

display(valid_snr_df.head())

if SAVE_TABLE and len(snr_df):
    table_path = SAVE_PATH / f"{today_str}_ASAP7_event_snr_all_sessions.csv"
    snr_df.to_csv(table_path, index=False)
    print(f"Saved: {table_path}")
    if len(error_df):
        error_path = SAVE_PATH / f"{today_str}_ASAP7_event_snr_errors.csv"
        error_df.to_csv(error_path, index=False)
        print(f"Saved: {error_path}")

[1/19] 826031_2026-01-30_15-04-02
  31 ROIs; 31 valid with >= 2 events
[2/19] 826031_2026-02-01_11-01-50
  32 ROIs; 32 valid with >= 2 events
[3/19] 826031_2026-02-02_10-23-53
  33 ROIs; 33 valid with >= 2 events
[4/19] 826031_2026-02-03_14-21-45
  37 ROIs; 37 valid with >= 2 events
[5/19] 826031_2026-02-04_12-15-34


In [ ]:
def make_session_summary(df):
    if len(df) == 0:
        return pd.DataFrame()

    metrics = [
        "event_snr_median",
        "event_snr_mean",
        "event_snr_p90",
        "event_snr_db_median",
        "event_snr_power_median",
        "event_amplitude_median_dff",
        "event_amplitude_p90_dff",
        "background_noise_sd_dff",
        "event_rate_per_min",
        "n_events",
    ]

    rows = []
    for keys, sub in df.groupby(["subject_id", "session_id", "dmd", "depth_um_below_pia"], dropna=False):
        row = dict(zip(["subject_id", "session_id", "dmd", "depth_um_below_pia"], keys))
        row["n_roi"] = len(sub)
        for m in metrics:
            row[f"{m}_median_across_roi"] = np.nanmedian(sub[m])
            row[f"{m}_mean_across_roi"] = np.nanmean(sub[m])
        rows.append(row)
    return pd.DataFrame(rows)


session_summary = make_session_summary(valid_snr_df)
display(session_summary)

if SAVE_TABLE and len(session_summary):
    session_summary_path = SAVE_PATH / f"{today_str}_ASAP7_event_snr_session_summary.csv"
    session_summary.to_csv(session_summary_path, index=False)
    print(f"Saved: {session_summary_path}")

## 3. Event SNR distributions: all ROIs, DMD1, DMD2

In [ ]:
def plot_event_snr_histograms(df, *, metric=SNR_PLOT_METRIC, bins=HIST_BINS):
    panels = [("All ROIs", df)] + [(dmd, df.query("dmd == @dmd")) for dmd in sorted(df["dmd"].dropna().unique())]

    values = df[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    if values.size == 0:
        raise ValueError(f"No finite values for {metric}")

    lo, hi = np.nanpercentile(values, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = np.nanmin(values), np.nanmax(values)
    edges = np.linspace(lo, hi, bins + 1)

    fig, axes = plt.subplots(1, len(panels), figsize=(5.4 * len(panels), 4.6), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for ax, (title, sub) in zip(axes, panels):
        vals = sub[metric].replace([np.inf, -np.inf], np.nan).dropna()
        ax.hist(vals, bins=edges, alpha=0.85)
        med = np.nanmedian(vals)
        ax.axvline(med, ls="--", lw=1.5, color="k", label=f"median = {med:.2f}")
        ax.set_title(f"{title}\nn={len(vals)} ROIs")
        ax.set_xlabel(metric)
        ax.legend(frameon=False, fontsize=10)
        ax.spines[["right", "top"]].set_visible(False)

    axes[0].set_ylabel("ROI count")
    fig.suptitle("ASAP7y detected-event SNR distributions")
    fig.tight_layout()
    return fig


fig = plot_event_snr_histograms(valid_snr_df)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_event_snr_histograms.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Event amplitude versus background noise

In [ ]:
def plot_event_amplitude_vs_background_noise(df, *, amplitude_metric="event_amplitude_median_dff"):
    fig, ax = plt.subplots(figsize=(7.5, 6.2))

    for dmd, sub in df.groupby("dmd", sort=True):
        ax.scatter(
            sub["background_noise_sd_dff"],
            sub[amplitude_metric],
            s=34,
            alpha=SCATTER_ALPHA,
            label=f"{dmd} (n={len(sub)})",
        )

    x = df["background_noise_sd_dff"].to_numpy()
    x = x[np.isfinite(x) & (x > 0)]
    if x.size == 0:
        raise ValueError("No finite positive background noise values.")
    xmin, xmax = np.nanpercentile(x, [1, 99])
    if xmin <= 0 or not np.isfinite(xmin):
        xmin = np.nanmin(x[x > 0])
    xx = np.geomspace(xmin, xmax * 1.3, 200)

    for snr in [2, 3, 5, 10, 20]:
        ax.plot(xx, snr * xx, ls="--", lw=0.9, color="0.35")
        ax.text(xx[-1], snr * xx[-1], f" SNR={snr}", va="center", fontsize=9, color="0.25")

    y = df[amplitude_metric].to_numpy()
    y = y[np.isfinite(y) & (y > 0)]
    ymin, ymax = np.nanpercentile(y, [1, 99])
    if ymin <= 0 or not np.isfinite(ymin):
        ymin = np.nanmin(y[y > 0])

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(xmin, xmax * 1.6)
    ax.set_ylim(ymin, ymax * 1.6)
    ax.set_xlabel("Non-event background noise SD (dF/F)")
    ax.set_ylabel("Detected event amplitude (dF/F)")
    ax.set_title("Detected event amplitude versus background variability")
    ax.legend(frameon=False)
    ax.spines[["right", "top"]].set_visible(False)
    fig.tight_layout()
    return fig


fig = plot_event_amplitude_vs_background_noise(valid_snr_df)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_event_amplitude_vs_background_noise.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. DMD/depth summary plots

In [ ]:
def plot_event_snr_by_dmd(df):
    metrics = [
        ("event_amplitude_median_dff", "median event amplitude (dF/F)"),
        ("background_noise_sd_dff", "non-event background noise SD (dF/F)"),
        (SNR_PLOT_METRIC, SNR_PLOT_METRIC),
        ("event_rate_per_min", "event rate/min"),
    ]

    fig, axes = plt.subplots(1, len(metrics), figsize=(5.0 * len(metrics), 4.8), constrained_layout=True)
    axes = np.atleast_1d(axes)

    rng = np.random.default_rng(0)
    for ax, (metric, ylabel) in zip(axes, metrics):
        groups = [sub[metric].dropna().to_numpy() for _, sub in df.groupby("dmd", sort=True)]
        labels = [dmd for dmd, _ in df.groupby("dmd", sort=True)]
        ax.boxplot(groups, labels=labels, showfliers=False)
        for i, vals in enumerate(groups, start=1):
            jitter = rng.normal(0, 0.035, size=len(vals))
            ax.scatter(np.full(len(vals), i) + jitter, vals, s=12, alpha=0.25)
        ax.set_ylabel(ylabel)
        ax.spines[["right", "top"]].set_visible(False)

    fig.suptitle("ROI-level event SNR components by DMD")
    return fig


fig = plot_event_snr_by_dmd(valid_snr_df)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_event_snr_by_dmd.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def plot_session_level_summary(session_summary):
    if len(session_summary) == 0:
        print("No session summary available.")
        return None

    metric = f"{SNR_PLOT_METRIC}_median_across_roi"
    fig, ax = plt.subplots(figsize=(7, 5))

    for dmd, sub in session_summary.groupby("dmd", sort=True):
        x = sub["depth_um_below_pia"].to_numpy()
        y = sub[metric].to_numpy()
        ax.scatter(x, y, s=70, alpha=0.8, label=dmd)
        # Connect points from same session where both DMDs exist.
    for sid, sub in session_summary.groupby("session_id"):
        if len(sub) > 1 and sub["depth_um_below_pia"].notna().all():
            sub = sub.sort_values("depth_um_below_pia")
            ax.plot(sub["depth_um_below_pia"], sub[metric], lw=0.8, alpha=0.35, color="0.5")

    ax.set_xlabel("Depth below pia (µm)")
    ax.set_ylabel(f"Session median {SNR_PLOT_METRIC}")
    ax.set_title("Session-level event SNR versus depth")
    ax.legend(frameon=False)
    ax.spines[["right", "top"]].set_visible(False)
    fig.tight_layout()
    return fig


fig = plot_session_level_summary(session_summary)
if SAVE_FIGURES and fig is not None:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_event_snr_depth_session_summary.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation guidance for slide 5

Suggested slide language:

> VIP dendritic ASAP7y traces contain detectable dF/F events whose amplitudes are several-fold above the local non-event background variability, with substantial ROI-to-ROI heterogeneity.

Recommended plot combination:

1. **Left:** example-session trace snippets for all ROIs, sorted by median event SNR.
2. **Right top:** event SNR histogram for all ROIs, DMD1, and DMD2.
3. **Right bottom:** event amplitude versus non-event background noise. Diagonal reference lines show SNR = 2, 3, 5, 10, 20.

Important caveats:

- This is a detected-event SNR, not a response SNR and not an absolute instrument sensitivity metric.
- The event detector is intentionally conservative. Changing `EVENT_THRESHOLD_SD`, `EVENT_PROMINENCE_SD`, or `EVENT_SMOOTH_MS` will change which events are included.
- For formal DMD/depth comparisons, prefer the `session_summary` table or a hierarchical model. ROI-level histograms are descriptive because ROIs within a session are not independent.
- If event rates are very high or traces are dominated by continuous slow changes, the event-excluded background may be underestimated. Use the sanity-check panel to visually inspect this.